<a href="https://colab.research.google.com/github/xozi/powerpkg/blob/main-py/problems/ee427_midterm.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install numpy==2.2.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 827.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 16.4/16.4 MB 39.3 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tensorflow 2.18.0 requires numpy<2.1.0,>=1.26.0, but you have numpy 2.2.3 which is incompatible.
numba 0.60.0 requires numpy<2.1,>=1.22, but you have numpy 2.2.3 which is incompatible.


In [2]:
import numpy as np

1) The diagram below shows a 3-bus AC power system with one generator.<br>
<img src='https://drive.google.com/uc?id=1_NUw3Sm6F0mxDUkCooL0TsOJF4AyIHLy'><br>
Generation and load information:<br>
𝐺1 = {223 MW, 77 MVAr}<br>
𝐿1 = {40 MW, 15 MVAr}<br>
𝐿2 = {110 MW, 25 MVAr}<br>
𝐿3 = {70 MW, 20 MVAr}<br>
Branch impedances are given in per unit.<br>
<br>
Assume:
- The generator at bus 1 has infinite capacity in both active power and reactive power.<br>
- The bus 1 is the reference bus, the angle at bus 1 is 0 and voltage magnitude at bus 1 is 1.02.<br>
- The MVA base is 100 MVA. <br>
<br>
Manually solve this AC power flow problem using the Newton-Raphson method for the first two iterations.

The follow problem has 3 buses, each with a load, and a generator on bus 1. Bus 1 is considered the slack bus with an angle of is 0 and magnitude of 1.02 pu. We also assume intial values of for angles and voltages for Bus 2 and 3 accordingly that will change over the iterations:

In [3]:
Sbase = 100e6
Vsp = np.array([1.02, 1.0, 1.0])
Angsp = np.array([0.0, 0.0, 0.0])
Psp = np.array([(223e6-40e6)/100e6, -110e6/100e6, -70e6/100e6])
Qsp = np.array([(77e6-15e6)/100e6, -25e6/100e6, -20e6/100e6])
busamount = 3

For the system we will need to make a nodal admittance matrix that defines the path impedeance between the loads. The self elements will be the self-admittances of the load, the off-diagonal elements will be mutual admittances. The graph shows no shunt impedeance.

In [4]:
Z_1to2 = 0.02+ 0.1j
Z_1to3 = 0.015+ 0.08j
Z_2to3 = 0.005+ 0.03j
Y_1to2 = 1/Z_1to2
Y_1to3 = 1/Z_1to3
Y_2to1 = Y_1to2
Y_2to3 = 1/Z_2to3
Y_3to1 = Y_1to3
Y_3to2 = Y_2to3
Ymatrix = np.array([[Y_1to2+Y_1to3, -Y_1to2, -Y_1to3],
                    [-Y_2to1, Y_2to1+Y_2to3, -Y_2to3],
                    [-Y_3to1, -Y_3to2, Y_3to1+Y_3to2]])
G = np.real(Ymatrix)
B = np.imag(Ymatrix)

To solve the Jacobians involved in the linear solving of the the change in voltage and angle, We define some functions used prior for the partitioned jacobian factors and a cleanup function that removes the slack bus from Newton-Rhapson Iteration:

In [5]:
def J1solve(Q, Anglediff, i, j):
    if i == j:
        return (-Q[i] - B[i,i] * (Vsp[i]**2))
    else:
        return (Vsp[i] * Vsp[j] * (G[i,j] * np.sin(Anglediff) - B[i,j] * np.cos(Anglediff)))

def J2solve(P, Anglediff, i, j):
    if i == j:
        return (P[i] + G[i,i] * (Vsp[i]**2))
    else:
        return (-Vsp[i] * (G[i,j] * np.cos(Anglediff) + B[i,j] * np.sin(Anglediff)))

def J3solve(P, Anglediff, i, j):
    if i == j:
        return (P[i] - G[i,i] * (Vsp[i]**2))
    else:
        return (Vsp[i] * Vsp[j] * (G[i,j] * np.cos(Anglediff) + B[i,j] * np.sin(Anglediff)))

def J4solve(Q, Anglediff, i, j):
    if i == j:
        return (Q[i] - B[i,i] * (Vsp[i]**2))
    else:
        return (Vsp[i] * (G[i,j] * np.sin(Anglediff) - B[i,j] * np.cos(Anglediff)))


def clean_slack(J1, J2, J3, J4, mismatch):
    J1r=J1[1:, 1:]
    J2r=J2[1:, 1:]
    J3r=J3[1:, 1:]
    J4r=J4[1:, 1:]
    J=np.block([[J1r, J2r], [J3r, J4r]])

    mismatch_reduced = np.concatenate([
        mismatch[0][1:],
        mismatch[1][1:]
    ])

    return J, mismatch_reduced

Lastly we do the Newton-Rhapson iteration for a total of 2 iterations, following a similar formula to Homework 4:

In [6]:
dP = np.zeros(len(Psp))
dQ = np.zeros(len(Qsp))

for iter in range(50):
    P = np.zeros(len(Psp))
    Q = np.zeros(len(Qsp))
    J1 = np.zeros((len(Psp), len(Psp)))
    J2 = np.zeros((len(Psp), len(Psp)))
    J3 = np.zeros((len(Psp), len(Psp)))
    J4 = np.zeros((len(Psp), len(Psp)))

    for i in range(busamount):
        for j in range(busamount):
            Anglediff = Angsp[i] - Angsp[j]
            P[i] += Vsp[i] * Vsp[j] * (G[i,j] * np.cos(Anglediff) + B[i,j] * np.sin(Anglediff))
            Q[i] += Vsp[i] * Vsp[j] * (G[i,j] * np.sin(Anglediff) - B[i,j] * np.cos(Anglediff))
        for k in range(busamount):
            Anglediff = Angsp[i] - Angsp[k]
            J1[i, k] += J1solve(Q, Anglediff, i, k)
            J2[i, k] += J2solve(P, Anglediff, i, k)
            J3[i, k] += J3solve(P, Anglediff, i, k)
            J4[i, k] += J4solve(Q, Anglediff, i, k)

    dP = Psp - P
    dQ = Qsp - Q

    mismatch = np.array([dP, dQ])

    J, mismatch = clean_slack(J1, J2, J3, J4, mismatch)

    if np.max(np.abs(mismatch)) < 1e-6:
        print(f"Converged in {iter+1} iterations")
        break

    X = np.linalg.solve(J, mismatch)

    n = len(X) // 2
    delta_theta = X[0:n]
    delta_V = X[n:]

    for i in range(1, len(Vsp)):
        Vsp[i] += delta_V[i-1] * Vsp[i]
        Angsp[i] += delta_theta[i-1]

    for i in range(busamount):
        if iter == 0:
            print("Voltage at iteration 1 at Bus " + str(i+1) + ": " + str(Vsp[i]) + " pu")
            print("Angle at iteration 1 at Bus " + str(i+1) + ": " + str(Angsp[i]*180/np.pi) + " deg")
        elif iter == 1:
            print("Voltage at iteration 2 at Bus " + str(i+1) + ": " + str(Vsp[i]) + " pu")
            print("Angle at iteration 2 at Bus " + str(i+1) + ": " + str(Angsp[i]*180/np.pi) + " deg")

for i in range(busamount):
    print("Final Voltage at Bus " + str(i+1) + ": " + str(Vsp[i]) + " pu")
    print("Final Angle at Bus " + str(i+1) + ": " + str(Angsp[i]*180/np.pi) + " deg")

Voltage at iteration 1 at Bus 1: 1.02 pu
Angle at iteration 1 at Bus 1: 0.0 deg
Voltage at iteration 1 at Bus 2: 0.9590357513638684 pu
Angle at iteration 1 at Bus 2: -2.011652539181907 deg
Voltage at iteration 1 at Bus 3: 0.9616536819608454 pu
Angle at iteration 1 at Bus 3: -1.6383955182956413 deg
Voltage at iteration 2 at Bus 1: 1.02 pu
Angle at iteration 2 at Bus 1: 0.0 deg
Voltage at iteration 2 at Bus 2: 0.9494552974075713 pu
Angle at iteration 2 at Bus 2: -3.896904740859279 deg
Voltage at iteration 2 at Bus 3: 0.9535322187239486 pu
Angle at iteration 2 at Bus 3: -3.4531441495367714 deg
Converged in 33 iterations
Final Voltage at Bus 1: 1.02 pu
Final Angle at Bus 1: 0.0 deg
Final Voltage at Bus 2: 0.9791541462013429 pu
Final Angle at Bus 2: -4.61215916243744 deg
Final Voltage at Bus 3: 0.9820583963950705 pu
Final Angle at Bus 3: -4.167753031465827 deg


The final calculated values were made for the first two iterations + final convergence to be tested against Powerworld. My final results from Powerworld showed: <br>
Bus 2: V= 0.9889 pu, Angle = -0.654 deg<br>
Bus 3: V = 0.9902 pu, Angle = -1.307 deg <br>
Which is a large discrepancy in the angle and voltage from my final calculate values. I looked through Powerworld for discrepancies after reviewing my intial values and more complex nodal admittance matrix and unforunately came up inconclusive what caused this difference. The Newton-Rhapson iteration worked fine in the previous example given in Homework 4.

2) A 12.47-kV power distribution feeder provides service to an unbalanced wye-connected load specified to be:<br>
Phase a: 1000kVA, 0.9 lagging power factor.<br>
Phase b: 800kVA, 0.9 lagging power factor.<br>
Phase c: 1100kVA, 0.9 lagging power factor.<br>
* Compute the initial load currents, assuming the loads are modeled as constant complex power.<br>
* Compute the initial load currents, assuming that 60% of the load is complex power, 25% constant current, and 15% constant impedance.<br>

The following problem begins with us calculating the intial load current assuming it is a constant complex power (PQ) wye-connected load. This means the power equations for them the currents are as follows:

$I_{PQ} = (\frac{S}{V_{LN}})^*$

Written in code this would be:

In [7]:
def toPolar(V):
    magnitude = np.abs(V)
    angle = np.degrees(np.angle(V))
    return magnitude, angle

def toRectangular(magnitude, angle, radians=False):
    if radians:
        return magnitude * np.exp(1j * angle)
    else:
        return magnitude * np.exp(1j * np.radians(angle))

def make_VLN_WyeSource(V, LN=True):
    v_angles = [0.0, -120.0 if len(V) >= 2 else 0.0, 120.0 if len(V) == 3 else 0.0]
    Vsource = []
    for v, angle in zip(V, v_angles):
        if LN:
            Vsource.append(toRectangular(v, angle))
        else:
            Vsource.append(toRectangular(v / np.sqrt(3), angle))
    return Vsource

voltage = make_VLN_WyeSource([12.47e3, 12.47e3, 12.47e3])
power_angle = np.acos(0.9);

S = [toRectangular(1000e3, power_angle, True),
    toRectangular(800e3, power_angle, True),
    toRectangular(1100e3, power_angle, True)]

I = [];
print("Initial Load Currents (Constant Power 100%):")
for i in range(3):
    I.append(np.conj(S[i] / voltage[i]))
    (I_mag, theta) = toPolar(I[i])
    print(f"Magnitude(amps): {I_mag}, Angle(deg): {theta}")


Initial Load Currents (Constant Power 100%):
Magnitude(amps): 80.19246190858058, Angle(deg): -25.841932763167126
Magnitude(amps): 64.15396952686447, Angle(deg): -145.8419327631671
Magnitude(amps): 88.21170809943865, Angle(deg): 94.15806723683286


For the distributed load, seperated by 60% complex power, 25% complex current, and 15% constant impedeance, the intial current can be calculated by begging by calculating constant power current from above and only taking 60% of it:

In [8]:
Ipq = []
print("Intial Load Current (Constant Power 60%):")
for i in range(3):
    Ipq.append(I[i]*0.6)
    (I_mag, theta) = toPolar(Ipq[i])
    print(f"Magnitude(amps): {I_mag}, Angle(deg): {theta}")

Intial Load Current (Constant Power 60%):
Magnitude(amps): 48.115477145148354, Angle(deg): -25.841932763167122
Magnitude(amps): 38.492381716118686, Angle(deg): -145.8419327631671
Magnitude(amps): 52.92702485966319, Angle(deg): 94.15806723683286


For the constant impedance (or the first iteration of it) composing 15% of the load, calculate the impedance first then the current after: <br>
$Z=(\frac{|V_{LN}^2|}{S^*}) \cdot \textrm{(% of PQ Load)}$ <br>
$I_Z=(\frac{V_{LN}}{Z}) \cdot \textrm{(% of Constant Impedance)}$ <br>

In [9]:
Z = []
Iz = []
print("Intial Load Current (Constant Impedance 15%):")
for i in range(3):
    Z.append((abs(voltage[i])**2/np.conj(S[i]))*0.6)
    Iz.append((voltage[i]/Z[i])*0.15)
    (I_mag, theta) = toPolar(Iz[i])
    print(f"Magnitude(amps): {I_mag}, Angle(deg): {theta}")

Intial Load Current (Constant Impedance 15%):
Magnitude(amps): 20.048115477145146, Angle(deg): -25.841932763167126
Magnitude(amps): 16.038492381716125, Angle(deg): -145.8419327631671
Magnitude(amps): 22.05292702485967, Angle(deg): 94.15806723683286


The constant current load of 25% is typically calculated by taking the magnitude of the calculated current under constant power and then iterate over the change in angle in the voltage.<br>

$I_L=|I_L| \cdot \textrm{(% of Constant Current) } ∠{\delta-\theta}$<br>

For the first iteration the angle doesn't change so only a simple multiplication of the constant current percentage is needed:<br>

In [10]:
IL = []
print("Intial Load Current (Constant Current 25%):")
for i in range(3):
    IL.append(I[i]*0.25)
    (I_mag, theta) = toPolar(IL[i])
    print(f"Magnitude(amps): {I_mag}, Angle(deg): {theta}")

Intial Load Current (Constant Current 25%):
Magnitude(amps): 20.048115477145146, Angle(deg): -25.841932763167126
Magnitude(amps): 16.038492381716118, Angle(deg): -145.8419327631671
Magnitude(amps): 22.052927024859663, Angle(deg): 94.15806723683286


The total combined load current is the sum of these intial current loads: <br>
$I_{sum}=I_{PQ} + I_Z + I_L$

In [11]:
I_sum = []
print("Intial Load Current (Total):")
for i in range(3):
    I_sum.append(Ipq[i] + Iz[i] + IL[i])
    (I_mag, theta) = toPolar(I_sum[i])
    print(f"Magnitude(amps): {I_mag}, Angle(deg): {theta}")


Intial Load Current (Total):
Magnitude(amps): 88.21170809943865, Angle(deg): -25.841932763167126
Magnitude(amps): 70.56936647955092, Angle(deg): -145.8419327631671
Magnitude(amps): 97.03287890938253, Angle(deg): 94.15806723683286


The results of the show increase in magnitude of intial load current due to the influence of the different load types.